# Embedding Models — HuggingFace for Code Semantic Search

## Problem Statement
Compare two embedding models for retrieving best-practice rules from buggy Python code descriptions, and determine whether phrasing strategy matters more than model choice.

## My Approach
* I expected all-mpnet-base-v2 to outperform MiniLM because it has a larger dimensionality (768 vs 384) and is generally considered more accurate. 
* My plan was to call the HuggingFace Inference API for both models, run cosine similarity manually, and compare which rule each bug description retrieves. 
* I also suspected phrasing the rules as natural language descriptions rather than formal rule titles would matter — but I hadn't tested it yet.

## What I Learned
* Embeddings turn meaning into geometry. Two sentences that describe the same bug — even with different words — land close together in vector space. That's the whole trick behind semantic search, and seeing it work on my own corpus made it click in a way reading about it never did.
* The HuggingFace Inference API URL changed. The old /models/<id> endpoint silently 404s for sentence-transformer models. 
* The correct route is /router.huggingface.co/hf-inference/models/.../pipeline/feature-extraction — and you need a token with "Make calls to Inference Providers" permission explicitly checked. 
* This cost me more debugging time than the actual ML work.
* Both models retrieved the same top rule for all 4 bugs. On this small, clean corpus the model size didn't matter — MiniLM and mpnet agreed completely. The real differentiator would show up on a larger, noisier rule set where semantically similar-sounding rules compete.

## Where I Got Stuck
* The HF API debugging loop ate most of my session. 
* Wrong endpoint → fixed endpoint → 401 because old token was cached in Windows env vars → hardcoded token to unblock → then restarting kernel as the actual fix. 
* Three separate failure modes that looked like the same error on the surface.

## What I'd Do Differently
* Run the phrasing experiment first, before the model comparison. The question "does how I phrase the document matter more than which model I use?" is more important than MiniLM vs mpnet - and the answer would have shaped which model I picked for ChromaDB persistence. * I also would have added similarity scores to the comparison table from the start instead of just printing the top match text — a score of 0.97 and a score of 0.61 are completely different situations even if the top match is the same rule.

In [1]:
from dotenv import load_dotenv
from huggingface_hub import login
import os
import requests
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import chromadb
import chromadb.utils.embedding_functions as embedding_functions

load_dotenv()

HF_TOKEN = os.getenv("HUGGING_FACE_API_KEY")

model_ids = {
    "MiniLM": "all-MiniLM-L6-v2",
    "MPNet": "all-mpnet-base-v2"
}



d:\Python\genai-engineering-portfolio\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_api_embeddings(texts, model_id):
    api_url = f"https://router.huggingface.co/hf-inference/models/sentence-transformers/{model_id}/pipeline/feature-extraction"

    headers = {"Authorization": f"Bearer {HF_TOKEN}"}
    
    payload = {
        "inputs": texts,
        "options": {"wait_for_model": True}  # ✅ eliminates the 503 retry loop
    }
    
    response = requests.post(api_url, headers=headers, json=payload)
    
    if response.status_code != 200:
        print(f"Error {response.status_code}: {response.text}")
        return None
    
    result = response.json()
    return np.array(result)  # shape: (N, 384) or (N, 768)

In [3]:
BUG_DESCRIPTIONS = [
    "This function uses a mutable default argument — a dict — which is shared across all calls",
    "This loop uses range(len(records)) instead of enumerate to iterate with index",
    "This file is opened with open() but never closed — no context manager used",
    "A variable called unused_var is assigned a string but never referenced again"
]

RULES = [
    "Avoid mutable default arguments in Python functions — use None and set inside the function body",
    "Use enumerate() instead of range(len()) when you need both index and value",
    "Always use context managers (with open()) to ensure file handles are closed",
    "Remove unused variables — they add noise and confuse readers",
    "Mutable default arguments are shared across all calls — this causes hidden state bugs",
    "range(len(x)) is a code smell in Python — enumerate gives you cleaner, more Pythonic code",
    "File handles not closed with context managers can cause resource leaks in long-running processes",
    "Dead code including unused bindings should be removed before review"
]

In [4]:

snip_vecs_mini = get_api_embeddings(BUG_DESCRIPTIONS, model_ids["MiniLM"])
rule_vecs_mini = get_api_embeddings(RULES, model_ids["MiniLM"])

snip_vecs_mpnet = get_api_embeddings(BUG_DESCRIPTIONS, model_ids["MPNet"])
rule_vecs_mpnet = get_api_embeddings(RULES, model_ids["MPNet"])


In [5]:
# 3. Compare and Table
def match_top(snippet_vecs, rule_vecs):
    sims = cosine_similarity(snippet_vecs, rule_vecs)
    return [RULES[np.argmax(row)] for row in sims]

comparison_df = pd.DataFrame({
    "Snippet": [s[:25] + "..." for s in BUG_DESCRIPTIONS],
    "MiniLM_Match": match_top(snip_vecs_mini, rule_vecs_mini),
    "MPNet_Match": match_top(snip_vecs_mpnet, rule_vecs_mpnet)
})

print(comparison_df)

                        Snippet  \
0  This function uses a muta...   
1  This loop uses range(len(...   
2  This file is opened with ...   
3  A variable called unused_...   

                                        MiniLM_Match  \
0  Avoid mutable default arguments in Python func...   
1  Use enumerate() instead of range(len()) when y...   
2  Always use context managers (with open()) to e...   
3  Remove unused variables — they add noise and c...   

                                         MPNet_Match  
0  Avoid mutable default arguments in Python func...  
1  Use enumerate() instead of range(len()) when y...  
2  Always use context managers (with open()) to e...  
3  Remove unused variables — they add noise and c...  


In [6]:
print(f"{'Bug':<25}| {'MiniLM Top Match':<25} | {'mpnet Top Match':<25}| Same?")
for index, row in comparison_df.iterrows():
    same= True if row['MiniLM_Match'] == row['MPNet_Match'] else False
    print(f"{row['Snippet'][:24]:<25}| {row['MiniLM_Match'][:24]:<25} | {row['MPNet_Match'][:24]:<25}| {same}")

Bug                      | MiniLM Top Match          | mpnet Top Match          | Same?
This function uses a mut | Avoid mutable default ar  | Avoid mutable default ar | True
This loop uses range(len | Use enumerate() instead   | Use enumerate() instead  | True
This file is opened with | Always use context manag  | Always use context manag | True
A variable called unused | Remove unused variables   | Remove unused variables  | True


In [7]:
# Phrasing Strategy: rule-as-written vs bug-description framing
RULES_AS_BUG_FRAMING = [
    "This code has a mutable default argument bug",
    "This code uses range(len()) instead of enumerate",
    "This code opens a file without a context manager",
    "This code has an unused variable",
    "This function's default argument is a mutable dict shared across calls",
    "This loop iterates with index using range(len()) — a Python code smell",
    "This file handle is never closed — potential resource leak",
    "This variable is assigned but never used — dead code",
]

rule_vecs_bug_framing = get_api_embeddings(RULES_AS_BUG_FRAMING, model_ids["MPNet"])

scores_original = cosine_similarity(snip_vecs_mpnet, rule_vecs_mpnet).max(axis=1)
scores_reframed = cosine_similarity(snip_vecs_mpnet, rule_vecs_bug_framing).max(axis=1)

print(f"{'Bug':<35} | {'Rule-as-written':>16} | {'Bug-framed':>10} | {'Winner':>8}")
print("-" * 80)
for i, bug in enumerate(BUG_DESCRIPTIONS):
    winner = "reframed" if scores_reframed[i] > scores_original[i] else "original"
    print(f"{bug[:34]:<35} | {scores_original[i]:>16.4f} | {scores_reframed[i]:>10.4f} | {winner:>8}")

Bug                                 |  Rule-as-written | Bug-framed |   Winner
--------------------------------------------------------------------------------
This function uses a mutable defau  |           0.6084 |     0.9241 | reframed
This loop uses range(len(records))  |           0.6491 |     0.6812 | reframed
This file is opened with open() bu  |           0.6453 |     0.8359 | reframed
A variable called unused_var is as  |           0.5259 |     0.6747 | reframed


In [11]:
# Create the persistent client
client = chromadb.PersistentClient(path="./chroma_db")

# Tell Chroma to use the API for all future queries
hf_ef = embedding_functions.HuggingFaceEmbeddingFunction(
    api_key=HF_TOKEN,
    model_name="sentence-transformers/all-mpnet-base-v2"
)

collection = client.get_or_create_collection(
    name="api_driven_rules", 
    embedding_function=hf_ef
)

# 3. Add to collection using the pre-computed embeddings
collection.add(
    embeddings=rule_vecs_mpnet.tolist(), # Convert numpy to list for Chroma
    documents=RULES,
    ids=[f"rule_{i}" for i in range(len(RULES))],
    metadatas=[{"type": "rule", "rule_id": i} for i in range(len(RULES))]
)

print(f"Collection count: {collection.count()} - Success using pre-computed vectors!")


Collection count: 8 - Success using pre-computed vectors!


In [12]:
# Fetch all data from the collection
results = collection.get()

# Iterate through IDs, Documents, and Metadatas together
for id, doc, meta in zip(results['ids'], results['documents'], results['metadatas']):
    print(f"ID: {id}")
    print(f"Content: {doc}")
    print(f"Metadata: {meta}")
    print("-" * 20)

ID: rule_0
Content: Avoid mutable default arguments in Python functions — use None and set inside the function body
Metadata: {'type': 'rule', 'rule_id': 0}
--------------------
ID: rule_1
Content: Use enumerate() instead of range(len()) when you need both index and value
Metadata: {'rule_id': 1, 'type': 'rule'}
--------------------
ID: rule_2
Content: Always use context managers (with open()) to ensure file handles are closed
Metadata: {'rule_id': 2, 'type': 'rule'}
--------------------
ID: rule_3
Content: Remove unused variables — they add noise and confuse readers
Metadata: {'type': 'rule', 'rule_id': 3}
--------------------
ID: rule_4
Content: Mutable default arguments are shared across all calls — this causes hidden state bugs
Metadata: {'type': 'rule', 'rule_id': 4}
--------------------
ID: rule_5
Content: range(len(x)) is a code smell in Python — enumerate gives you cleaner, more Pythonic code
Metadata: {'type': 'rule', 'rule_id': 5}
--------------------
ID: rule_6
Content: File

 **Refactored solution**

In [13]:
def match_top_with_scores(snippet_vecs, rule_vecs):
    sims = cosine_similarity(snippet_vecs, rule_vecs)
    top_indices = np.argmax(sims, axis=1)
    top_scores = np.max(sims, axis=1)
    return [RULES[i] for i in top_indices], top_scores

mini_matches, mini_scores = match_top_with_scores(snip_vecs_mini, rule_vecs_mini)
mpnet_matches, mpnet_scores = match_top_with_scores(snip_vecs_mpnet, rule_vecs_mpnet)

print(f"{'Bug':<30} | {'MiniLM Match':<45} | {'Score':>6} | {'MPNet Match':<45} | {'Score':>6} | Same?")
print("-" * 150)
for i, bug in enumerate(BUG_DESCRIPTIONS):
    same = "✅" if mini_matches[i] == mpnet_matches[i] else "❌"
    print(f"{bug[:29]:<30} | {mini_matches[i][:44]:<45} | {mini_scores[i]:.4f} | {mpnet_matches[i][:44]:<45} | {mpnet_scores[i]:.4f} | {same}")

Bug                            | MiniLM Match                                  |  Score | MPNet Match                                   |  Score | Same?
------------------------------------------------------------------------------------------------------------------------------------------------------
This function uses a mutable   | Avoid mutable default arguments in Python fu  | 0.6686 | Avoid mutable default arguments in Python fu  | 0.6084 | ✅
This loop uses range(len(reco  | Use enumerate() instead of range(len()) when  | 0.7606 | Use enumerate() instead of range(len()) when  | 0.6491 | ✅
This file is opened with open  | Always use context managers (with open()) to  | 0.7330 | Always use context managers (with open()) to  | 0.6453 | ✅
A variable called unused_var   | Remove unused variables — they add noise and  | 0.5377 | Remove unused variables — they add noise and  | 0.5259 | ✅
